In [4]:
from src.evaluation.sealqa_scorer import score_sealqa
import pickle
import pandas as pd
import json
from tqdm import tqdm

with open("../results/browsecomp_eval_results_noskill.pkl", 'rb') as f:
    all_results = pickle.load(f)

# Keep only examples that have trace.output.final_answer
results = [
    r for r in all_results
    if r.trace and r.trace.output and r.trace.output.final_answer
]

In [5]:
len(results)

144

In [6]:
# sample_data= results[:80]

In [7]:
from concurrent.futures import ThreadPoolExecutor, as_completed

def score_one(result):
    score = score_sealqa(
        question=result.question,
        ground_truth=result.ground_truth,
        predicted=result.trace.output.final_answer,
    )
    return result, score

scored_results = []
with ThreadPoolExecutor(max_workers=5) as executor:
    futures = {executor.submit(score_one, r): i for i, r in enumerate(results[:128])}
    for future in tqdm(as_completed(futures), total=len(futures), desc="Scoring"):
        try:
            result, score = future.result()
            result.score = score
            scored_results.append(result)
        except Exception as e:
            # Optionally: print(f"Skipped index {futures[future]}: {e}")
            pass

# Only scored_results have .score; use len(scored_results) for avg
scores = [r.score for r in scored_results]
sum(scores) / len(scores) if scores else 0

Scoring: 100%|██████████| 128/128 [00:00<00:00, 424.83it/s]


0.40625